<a href="https://colab.research.google.com/github/raisharad/GenerativeAIandAgenticAI/blob/main/June20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
# Install the tools we need

!pip install sentencepiece datasets
!pip install deep-translator

In [19]:
import torch
torch.__version__

'2.11.0+cu128'

In [20]:
# Import everything
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm
import math, random, os, time

# Use GPU if available (makes training much faster)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Tip: go to Runtime → Change runtime type → T4 GPU for faster training")

random.seed(42)
torch.manual_seed(42)
print("Ready!")



Using: cuda
GPU: Tesla T4
Ready!


"I go to school"

In [21]:
from datasets import load_dataset

print("Loading 5,000 sentence pairs from ai4bharat/samanantar ...")
print("(This downloads from the internet — takes ~1 minute)")

ds = load_dataset(
    "ai4bharat/samanantar",
    "hi",                    # "hi" = Hindi
    split="train",
    streaming=True           # stream so we only download what we need
)

PAIRS = []   # will store (Hindi, English) tuples
for row in ds:
    bn = row["tgt"].strip()   # Hindi sentence
    en = row["src"].strip()   # English sentence
    if bn and en:
        PAIRS.append((bn, en))
    if len(PAIRS) >= 5000:
        break

print(f"Loaded {len(PAIRS)} sentence pairs")
print()
print("Examples from the dataset:")
print("-" * 55)
for hi_text, en_text in PAIRS[:5]:
    print(f"  Hindi  : {hi_text}")
    print(f"  English: {en_text}")
    print()

Loading 5,000 sentence pairs from ai4bharat/samanantar ...
(This downloads from the internet — takes ~1 minute)
Loaded 5000 sentence pairs

Examples from the dataset:
-------------------------------------------------------
  Hindi  : आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।
  English: However, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles

  Hindi  : और जो शख्स (अपने आमाल का) बदला दुनिया ही में चाहता है तो ख़ुदा के पास दुनिया व आख़िरत दोनों का अज्र मौजूद है और ख़ुदा तो हर शख्स की सुनता और सबको देखता है
  English: Whosoever desires the reward of the world, with Allah is the reward of the world and of the Everlasting Life. Allah is the Hearer, the Seer.

  Hindi  : जैव-मंडल में कीड़ों का मूल्य बहुत है, क्योंकि प्रजातियों की समृद्धि के मामले में उनकी संख्या अन्य जीव समूहों से ज़्यादा है।
  English: T

In [27]:
from deep_translator import GoogleTranslator

text = "आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।"

translation = GoogleTranslator(
    source='hi',
    target='en'
).translate(text)

print(translation)

Paes, who partnered with Australian Paul Henley, could only reach the quarterfinals in Miami as he was defeated by Bhupathi and Knowles in this round.


Step 3.1: Tokenization — splitting sentences into pieces
A neural network cannot read text directly. We must convert each sentence into a list of numbers.

But we can't just assign one number per word — Hindi has millions of word forms. Instead we use BPE (Byte Pair Encoding): split words into common sub-pieces.

In [23]:
# Train tokenizers — one for Hindi, one for English
# Think of this like building a "dictionary of pieces"

PAD_ID = 0   # padding (to make all sentences same length)
BOS_ID = 1   # beginning of sentence <s>
EOS_ID = 2   # end of sentence </s>

os.makedirs("spm", exist_ok=True)

def train_tokenizer(texts, save_name, vocab_size, lang="en"):
    """Train a BPE tokenizer on a list of sentences."""
    spm.SentencePieceTrainer.Train(
        sentence_iterator=iter(texts),
        model_prefix=save_name,
        vocab_size=vocab_size,
        character_coverage=0.9999 if lang=="hi" else 0.9995,
        model_type="bpe",
        pad_id=PAD_ID, bos_id=BOS_ID, eos_id=EOS_ID, unk_id=3,
        pad_piece="<pad>", bos_piece="<s>",
        eos_piece="</s>", unk_piece="<unk>",
    )

hi_sentences = [p[0] for p in PAIRS]
en_sentences = [p[1] for p in PAIRS]

train_tokenizer(hi_sentences, "spm/hindi",   vocab_size=4000, lang="hi")
train_tokenizer(en_sentences, "spm/english", vocab_size=4000, lang="en")
print("Tokenizers trained!")

# Load them back
hi_sp = spm.SentencePieceProcessor(); hi_sp.Load("spm/hindi.model")
en_sp = spm.SentencePieceProcessor(); en_sp.Load("spm/english.model")

# Show what tokenization looks like
example_hi = PAIRS[0][0]
example_en = PAIRS[0][1]
print()
print("Example Hindi sentence:")
print(f"  Text  : {example_hi}")
print(f"  Pieces: {hi_sp.EncodeAsPieces(example_hi)}")
print(f"  IDs   : {hi_sp.EncodeAsIds(example_hi)}")
print()
print("Example English sentence:")
print(f"  Text  : {example_en}")
print(f"  Pieces: {en_sp.EncodeAsPieces(example_en)}")
print(f"  IDs   : {en_sp.EncodeAsIds(example_en)}")

Tokenizers trained!

Example Hindi sentence:
  Text  : आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।
  Pieces: ['▁आस्ट्रेलिया', '▁के', '▁प', 'ाल', '▁हे', 'न', 'ली', '▁के', '▁साथ', '▁जोड़ी', '▁बनाने', '▁वाले', '▁प', 'ेस', '▁म', 'िया', 'मी', '▁में', '▁क्व', 'ार्ट', 'र', 'फाइ', 'नल', '▁तक', '▁ही', '▁पहुंच', '▁सके', '▁क्योंकि', '▁इस', '▁दौर', '▁में', '▁उन्हें', '▁भ', 'ूप', 'ति', '▁और', '▁नो', 'ल्स', '▁ने', '▁हरा', 'या', '▁था', '।']
  IDs   : [3469, 10, 8, 51, 858, 3840, 123, 10, 154, 2707, 717, 398, 8, 1236, 7, 65, 500, 19, 1868, 2239, 3836, 3528, 1468, 245, 157, 659, 1360, 789, 56, 460, 19, 453, 38, 228, 70, 32, 3559, 1695, 48, 2861, 15, 151, 3861]

Example English sentence:
  Text  : However, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles
  Pieces: ['▁However', ',', '▁P', 'a', 'es', ',', '▁who', '▁was', '

In [24]:
PAD_ID = 0   # padding token
BOS_ID = 1   # beginning of sentence
EOS_ID = 2   # end of sentence

os.makedirs("spm", exist_ok=True)

def train_tokenizer(texts, save_name, vocab_size, lang="en"):
    """Train a BPE tokenizer on a list of sentences."""
    spm.SentencePieceTrainer.Train(
        sentence_iterator=iter(texts),
        model_prefix=save_name,
        vocab_size=vocab_size,
        character_coverage=0.9999 if lang=="hi" else 0.9995,
        model_type="bpe",
        pad_id=PAD_ID, bos_id=BOS_ID, eos_id=EOS_ID, unk_id=3,
        pad_piece="<pad>", bos_piece="<s>",
        eos_piece="</s>", unk_piece="<unk>",
    )

hi_sentences = [p[0] for p in PAIRS]
en_sentences = [p[1] for p in PAIRS]

# Fix: Increase vocab_size for Hindi tokenizer
train_tokenizer(hi_sentences, "spm/hindi",   vocab_size=4000, lang="hi")
train_tokenizer(en_sentences, "spm/english", vocab_size=4000, lang="en")

print("Both tokenizers trained!")
print()
print("Hindi coverage used  :", 0.9999, "← correct for Devanagari")
print("English coverage used:", 0.9995, "← correct")

Both tokenizers trained!

Hindi coverage used  : 0.9999 ← correct for Devanagari
English coverage used: 0.9995 ← correct
